# Lesson 24: Search, planning and classical AI

*Worked solutions for [the lesson page](../../lessons/24-classical-ai.html). The exercise version without solutions is `../24-classical-ai.ipynb`.*

The half of AI that is not learned from data: search mazes and road maps, find routes with A*, schedule with a constraint solver, play games with minimax and Monte Carlo tree search, reason with rules and probabilities, and see how AlphaZero and LLM agents combine these ideas with learned models.

**Runs on:** Any computer (CPU), offline. Uses only the Python standard library, NumPy and scikit-learn; every cell finishes in a few seconds.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy scikit-learn

## Problems as search

A problem has **states**, an **initial state**, **actions** with a transition model, a **goal test** and a **path cost**. In code, actions and transitions become one **successor function** that lists the next states and the cost of reaching each. Here is a maze in that form.

In [ ]:
MAZE = """\
S.......#.
.######.#.
......#...
.####.#.#.
.#G...#.#.
.######.#.
..........
"""

def find(grid, ch):
    return next((r, c) for r, row in enumerate(grid)
                for c, x in enumerate(row) if x == ch)

def neighbours(state, grid):
    """Successor function: yield (next_state, step_cost) for every legal move."""
    r, c = state
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:      # up, down, left, right
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] != "#":
            yield (nr, nc), 1

def show(grid, path):
    rows = [list(row) for row in grid]
    for r, c in path[1:-1]:
        rows[r][c] = "*"
    print("\n".join("".join(row) for row in rows))

grid = MAZE.splitlines()
start, goal = find(grid, "S"), find(grid, "G")
print("start", start, "goal", goal)
print("moves from start:", list(neighbours(start, grid)))

### Breadth-first and depth-first search

BFS takes states from the front of the frontier (a queue), so the first path it finds has the fewest steps. DFS takes them from the back (a stack) and dives as deep as it can. The `parent` dictionary records every state already reached, so neither expands a state twice.

In [ ]:
from collections import deque

def reconstruct(parent, s):
    path = []
    while s is not None:
        path.append(s)
        s = parent[s]
    return path[::-1]

def uninformed_search(start, is_goal, successors, kind="bfs"):
    frontier = deque([start])
    parent = {start: None}                # also the set of states already reached
    expanded = 0
    while frontier:
        s = frontier.popleft() if kind == "bfs" else frontier.pop()   # FIFO or LIFO
        expanded += 1
        if is_goal(s):
            return reconstruct(parent, s), expanded
        for s2, _ in successors(s):
            if s2 not in parent:
                parent[s2] = s
                frontier.append(s2)
    return None, expanded

moves = lambda s: neighbours(s, grid)
for kind in ["bfs", "dfs"]:
    path, n = uninformed_search(start, lambda s: s == goal, moves, kind)
    print(f"{kind}: {len(path) - 1} steps, {n} states expanded")
    show(grid, path)

### Uniform-cost search (Dijkstra's algorithm)

UCS always expands the frontier state with the lowest path cost so far, $g(n)$, using a priority queue. It must test for the goal when a state is **expanded**, not when it is generated, or it can return a more expensive route. The map has seven field sites and a 30 km mountain road from the depot straight to the ridge.

In [ ]:
import heapq, math

# Field sites with their (x, y) position in km, and roads with their length in km
SITES = {"depot": (0, 0), "harbour": (12, 4), "mill": (5, 9), "ridge": (14, 14),
         "lake": (22, 8), "airfield": (8, 18), "observatory": (24, 19)}
ROADS = [("depot", "harbour", 14), ("depot", "mill", 11), ("depot", "ridge", 30),
         ("harbour", "lake", 12), ("harbour", "ridge", 11), ("mill", "ridge", 11),
         ("mill", "airfield", 10), ("ridge", "observatory", 12),
         ("lake", "observatory", 12), ("airfield", "observatory", 17)]
graph = {s: [] for s in SITES}
for a, b, km in ROADS:
    graph[a].append((b, km))
    graph[b].append((a, km))

def uniform_cost_search(start, is_goal, successors):
    frontier = [(0, start)]               # a priority queue ordered by path cost g
    parent, g = {start: None}, {start: 0}
    expanded = 0
    while frontier:
        cost, s = heapq.heappop(frontier)
        if cost > g[s]:
            continue                      # stale: a cheaper route was found later
        expanded += 1
        if is_goal(s):                    # test when expanded, not when generated
            return reconstruct(parent, s), cost, expanded
        for s2, step in successors(s):
            new = cost + step
            if s2 not in g or new < g[s2]:
                g[s2], parent[s2] = new, s
                heapq.heappush(frontier, (new, s2))
    return None, math.inf, expanded

roads = lambda s: graph[s]
at_obs = lambda s: s == "observatory"
path, _ = uninformed_search("depot", at_obs, roads, "bfs")
km = sum(dict(graph[a])[b] for a, b in zip(path, path[1:]))
print("BFS:", " -> ".join(path), f"({km} km)")
path, km, n = uniform_cost_search("depot", at_obs, roads)
print("UCS:", " -> ".join(path), f"({km} km, {n} expanded)")

### Hands-on: when steps have different costs

1. Write `weighted_neighbours(state, grid)`, where stepping onto mud `~` costs 5.
2. Run BFS and UCS on the map below, and print each path and its cost.
3. Stretch: implement iterative deepening and check that it finds a 12-step path through the first maze.

```
S....~....
.....~....
.....~....
.....~...G
..........
```

You are done when BFS reports 12 steps costing 16 and UCS reports 14 steps costing 14.

#### Solution

A weighted successor function, then BFS and UCS on the mud map, then iterative deepening for the stretch.

In [ ]:
MUD = """\
S....~....
.....~....
.....~....
.....~...G
..........
"""

def weighted_neighbours(state, grid):
    """Like neighbours(), but stepping into mud costs 5."""
    for (r, c), _ in neighbours(state, grid):
        yield (r, c), 5 if grid[r][c] == "~" else 1

mud = MUD.splitlines()
s, t = find(mud, "S"), find(mud, "G")
moves = lambda p: weighted_neighbours(p, mud)
cost_of = lambda path: sum(5 if mud[r][c] == "~" else 1 for r, c in path[1:])

path, _ = uninformed_search(s, lambda p: p == t, moves, "bfs")
print(f"BFS: {len(path) - 1} steps, cost {cost_of(path)}")
show(mud, path)
path, cost, _ = uniform_cost_search(s, lambda p: p == t, moves)
print(f"UCS: {len(path) - 1} steps, cost {cost}")
show(mud, path)

# Stretch: iterative deepening, depth-first search with a growing depth limit
def depth_limited(path, is_goal, successors, limit):
    if is_goal(path[-1]):
        return path
    if limit == 0:
        return None
    for s2, _ in successors(path[-1]):
        if s2 not in path:                       # never revisit a state on this path
            found = depth_limited(path + [s2], is_goal, successors, limit - 1)
            if found:
                return found
    return None

def iterative_deepening(start, is_goal, successors, max_depth=100):
    for limit in range(max_depth + 1):
        found = depth_limited([start], is_goal, successors, limit)
        if found:
            return found

path = iterative_deepening(start, lambda p: p == goal, lambda p: neighbours(p, grid))
print("IDS on the first maze:", len(path) - 1, "steps")

**What to observe.** Every 12-step route has to cross the mud, and BFS cannot tell the difference. UCS pays two extra steps to go round it. Iterative deepening repeats the shallow levels many times, but each level is so much bigger than the ones before it that the repeated work adds little.

## Informed search

A **heuristic** $h(n)$ estimates the cost from $n$ to the goal. It is **admissible** if it never overestimates, $h(n) \le h^*(n)$, and **consistent** if $h(n) \le c(n, n') + h(n')$ for every successor $n'$. Relaxed problems give admissible heuristics: delete the maze walls and you get the Manhattan distance.

### A*, greedy and weighted A*

A* expands the state with the lowest $f(n) = g(n) + h(n)$. Greedy best-first search uses $f = h$ alone; weighted A* uses $f = g + w\,h$ with $w > 1$. One function covers all of them, and uniform-cost search too.

In [ ]:
def best_first(start, is_goal, successors, h=lambda s: 0, wg=1, wh=1):
    """Expand the node with the lowest f = wg*g + wh*h.
    wh=0 is uniform-cost search, wg=wh=1 is A*, wg=0 is greedy best-first."""
    frontier = [(wh * h(start), 0, start)]
    parent, g = {start: None}, {start: 0}
    expanded = 0
    while frontier:
        _, cost, s = heapq.heappop(frontier)
        if cost > g[s]:
            continue
        expanded += 1
        if is_goal(s):
            return reconstruct(parent, s), cost, expanded
        for s2, step in successors(s):
            new = cost + step
            if s2 not in g or new < g[s2]:
                g[s2], parent[s2] = new, s
                heapq.heappush(frontier, (wg * new + wh * h(s2), new, s2))
    return None, math.inf, expanded

# A 20 x 40 open field with a cup-shaped wall between start and goal
rows = [["."] * 40 for _ in range(20)]
for r in range(4, 16):
    rows[r][28] = "#"                      # the back of the cup
for c in range(18, 29):
    rows[4][c] = rows[15][c] = "#"         # its two arms
rows[10][4], rows[10][36] = "S", "G"
field = ["".join(row) for row in rows]
s, t = find(field, "S"), find(field, "G")
manhattan = lambda p: abs(p[0] - t[0]) + abs(p[1] - t[1])
moves = lambda p: neighbours(p, field)
done = lambda p: p == t

path, n = uninformed_search(s, done, moves, "bfs")
print(f"{'BFS':12} cost {len(path) - 1}  expanded {n:3}")
variants = [("UCS", 1, 0), ("A*", 1, 1), ("weighted A*", 1, 2), ("greedy", 0, 1)]
for name, wg, wh in variants:
    path, cost, n = best_first(s, done, moves, manhattan, wg, wh)
    print(f"{name:12} cost {cost}  expanded {n:3}")

Now the road map, with straight-line distance as the heuristic. The first line checks that the heuristic really is admissible: no road is shorter than the straight line it spans.

In [ ]:
def straight_line(site):
    return math.dist(SITES[site], SITES["observatory"])

# Admissible and consistent: no road is shorter than the straight line it spans
print(all(km >= math.dist(SITES[a], SITES[b]) for a, b, km in ROADS))
for name, wg, wh in [("UCS", 1, 0), ("A*", 1, 1), ("greedy", 0, 1)]:
    path, km, n = best_first("depot", at_obs, roads, straight_line, wg, wh)
    print(f"{name:7}{' -> '.join(path)} ({km} km, {n} expanded)")

## Constraint satisfaction

A CSP has **variables**, a **domain** of values for each and **constraints** on which combinations are allowed. Backtracking assigns one variable at a time and undoes the last choice when a constraint breaks. **Forward checking** prunes the neighbours' domains after each assignment, and **MRV** assigns the variable with the fewest values left first. To start, a week of maintenance visits.

In [ ]:
def forward_check(var, a, assignment, domains, cons):
    """Remove values that clash with var=a from unassigned neighbours' domains."""
    pruned = dict(domains)
    for y, ok in cons[var]:
        if y not in assignment:
            pruned[y] = [b for b in domains[y] if ok(a, b)]
            if not pruned[y]:
                return None               # a neighbour has no value left: dead end
    return pruned

def backtrack(domains, cons, assignment=None, fc=True, stats=None):
    assignment = {} if assignment is None else assignment
    if len(assignment) == len(domains):
        return dict(assignment)
    # Most constrained variable first: the one with the fewest values left (MRV)
    unassigned = [v for v in domains if v not in assignment]
    var = min(unassigned, key=lambda v: len(domains[v]))
    for a in domains[var]:
        if all(ok(a, assignment[y]) for y, ok in cons[var] if y in assignment):
            stats["assignments"] += 1
            assignment[var] = a
            pruned = domains
            if fc:
                pruned = forward_check(var, a, assignment, domains, cons)
            if pruned is not None:
                result = backtrack(pruned, cons, assignment, fc, stats)
                if result:
                    return result
            del assignment[var]           # undo, then try the next value
    return None

JOBS = {"belt P-117": "Ana", "calibrate P-117": "Ana", "inspect P-310": "Ana",
        "firmware P-204": "Ben", "optics P-204": "Ben", "battery P-310": "Ben"}
DAYS = ["Mon", "Tue", "Wed"]
domains = {job: [0, 1, 2] for job in JOBS}
domains["inspect P-310"] = [2]            # site P-310 only opens on Wednesday
cons = {job: [] for job in JOBS}

def constrain(x, y, ok):                  # ok(a, b) says whether x=a, y=b is allowed
    cons[x].append((y, ok))
    cons[y].append((x, lambda b, a: ok(a, b)))

for x in JOBS:
    for y in JOBS:
        if x < y and JOBS[x] == JOBS[y]:          # same technician,
            constrain(x, y, lambda a, b: a != b)  # so different days
constrain("belt P-117", "calibrate P-117", lambda a, b: a < b)   # belt first
constrain("firmware P-204", "optics P-204", lambda a, b: a < b)
constrain("inspect P-310", "battery P-310", lambda a, b: a == b) # shared van

stats = {"assignments": 0}
schedule = backtrack(domains, cons, stats=stats)
for d, day in enumerate(DAYS):
    print(day, sorted(job for job, v in schedule.items() if v == d))
print(stats)

N-queens: place $n$ queens on an $n \times n$ board so that no two share a row, column or diagonal. Compare the work with and without forward checking.

In [ ]:
def queens(n):
    # Variable c is a column; its value is the row of the queen in that column
    domains = {c: list(range(n)) for c in range(n)}
    cons = {c: [(c2, lambda r1, r2, d=abs(c - c2): r1 != r2 and abs(r1 - r2) != d)
                for c2 in range(n) if c2 != c] for c in range(n)}
    return domains, cons

for n in [8, 12, 16]:
    for fc in [False, True]:
        stats = {"assignments": 0}
        backtrack(*queens(n), fc=fc, stats=stats)
        count = stats["assignments"]
        print(f"n={n:3}  forward checking={fc!s:5}  assignments={count:6}")
stats = {"assignments": 0}
backtrack(*queens(100), stats=stats)
print(f"n=100  forward checking=True   assignments={stats['assignments']:6}")

### Hands-on: solve a Sudoku as a constraint problem

1. Make one variable per cell, keyed by `(row, col)`, with the given digit or 1 to 9 as its domain. The puzzle, row by row with 0 for a blank: `530070000 600195000 098000060 800060003 400803001 700020006 060000280 000419005 000080079`.
2. Add a not-equal constraint between every pair of **peers**: cells in the same row, column or 3 by 3 box. Each cell has 20.
3. Solve it with `backtrack`, print the grid, and compare the assignments with `fc=True` and `fc=False`. Stretch: try `800000000 003600000 070090200 050007000 000045700 000100030 001000068 008500010 090000400`.

You are done when the puzzle takes exactly 81 assignments with forward checking, meaning no backtracking, and thousands without.

#### Solution

Eighty-one variables, a not-equal constraint between every pair of peers, and the `backtrack` solver from the lesson unchanged.

In [ ]:
PUZZLE = ("530070000" "600195000" "098000060"
          "800060003" "400803001" "700020006"
          "060000280" "000419005" "000080079")
HARD = ("800000000" "003600000" "070090200"
        "050007000" "000045700" "000100030"
        "001000068" "008500010" "090000400")
cells = [(r, c) for r in range(9) for c in range(9)]

def peers(a, b):
    same_box = (a[0] // 3, a[1] // 3) == (b[0] // 3, b[1] // 3)
    return a != b and (a[0] == b[0] or a[1] == b[1] or same_box)

different = lambda x, y: x != y
sudoku_cons = {a: [(b, different) for b in cells if peers(a, b)] for a in cells}
print("peers per cell:", len(sudoku_cons[(0, 0)]))

def solve_sudoku(puzzle, fc=True):
    doms = {(r, c): [int(puzzle[9 * r + c])] if puzzle[9 * r + c] != "0"
            else list(range(1, 10)) for r, c in cells}
    stats = {"assignments": 0}
    return backtrack(doms, sudoku_cons, fc=fc, stats=stats), stats["assignments"]

solution, n = solve_sudoku(PUZZLE)
for r in range(9):
    print(" ".join(str(solution[(r, c)]) for c in range(9)))
print("forward checking:", n, "assignments")
print("plain backtracking:", solve_sudoku(PUZZLE, fc=False)[1], "assignments")
print("hard puzzle, forward checking:", solve_sudoku(HARD)[1], "assignments")

**What to observe.** MRV places the givens first, then forward checking does what you do with a pencil: "this cell can only be a 4". On the easy puzzle that alone is enough (81 assignments, no backtracking). The harder puzzle needs real search, about ten thousand assignments, and still finishes in a fraction of a second. Stronger propagation, such as arc consistency, would cut that further.

### The industrial versions

Real scheduling, routing and configuration problems go to SAT, SMT, constraint programming and integer programming solvers, such as Z3 and the CP-SAT solver in Google's OR-Tools. Your job is to model the problem; the solver does the search.

## Adversarial search: playing games

In a two-player, zero-sum game the search has to consider the opponent's replies as well as your own moves.

### Minimax and alpha-beta pruning

Minimax assumes the opponent always picks the reply that is worst for you. Alpha-beta pruning tracks $\alpha$ (what X can already guarantee) and $\beta$ (what O can already hold X to), and skips the rest of a position's moves once $\alpha \ge \beta$: the same answer from far fewer nodes. The cached `value` function is a transposition table that solves each distinct position once.

In [ ]:
from functools import lru_cache

LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6),
         (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]

def winner(b):
    for i, j, k in LINES:
        if b[i] != "." and b[i] == b[j] == b[k]:
            return b[i]
    return None

def legal(b):      return [i for i, v in enumerate(b) if v == "."]
def play(b, i, p): return b[:i] + p + b[i + 1:]
def other(p):      return "O" if p == "X" else "X"

def minimax(b, player, alpha=-2, beta=2, prune=True, stats=None):
    """Value of board b for X (+1 win, 0 draw, -1 loss) with `player` to move."""
    stats["nodes"] += 1
    w = winner(b)
    if w or "." not in b:
        return {"X": 1, "O": -1, None: 0}[w]
    values = []
    for i in legal(b):
        v = minimax(play(b, i, player), other(player), alpha, beta, prune, stats)
        values.append(v)
        if player == "X":
            alpha = max(alpha, v)         # X can already guarantee at least alpha
        else:
            beta = min(beta, v)           # O can already hold X to at most beta
        if prune and alpha >= beta:
            break                 # the opponent will never allow this line
    return max(values) if player == "X" else min(values)

for prune in [False, True]:
    stats = {"nodes": 0}
    v = minimax("." * 9, "X", prune=prune, stats=stats)
    print(f"alpha-beta={prune!s:5}  value={v}  nodes searched={stats['nodes']:,}")

# A transposition table: lru_cache solves each distinct position only once
@lru_cache(maxsize=None)
def value(b, player):
    w = winner(b)
    if w or "." not in b:
        return {"X": 1, "O": -1, None: 0}[w]
    vals = [value(play(b, i, player), other(player)) for i in legal(b)]
    return max(vals) if player == "X" else min(vals)

def perfect_move(b, player):
    pick = max if player == "X" else min
    return pick(legal(b), key=lambda i: value(play(b, i, player), other(player)))

print("value", value("." * 9, "X"), "from",
      value.cache_info().currsize, "distinct positions")

### Monte Carlo tree search

Each iteration has four steps: **select** a path down the tree by the UCT score $W_i/N_i + c\sqrt{\ln N / N_i}$, **expand** one untried move, **simulate** a random game to the end, and **backpropagate** the result to every node on the path. After all iterations, play the most visited move. The matches below take a few seconds.

In [ ]:
import random

class Node:
    def __init__(self, board, player, parent=None, move=None):
        self.board, self.player, self.parent, self.move = board, player, parent, move
        self.children = []
        self.untried = [] if winner(board) else legal(board)
        # score is from the point of view of the player who moved INTO this node
        self.visits, self.score = 0, 0.0

def uct(child, parent_visits, c):
    return (child.score / child.visits
            + c * math.sqrt(math.log(parent_visits) / child.visits))

def mcts(board, player, iterations=500, c=1.4, rng=random):
    root = Node(board, player)
    for _ in range(iterations):
        node = root
        # 1. Selection: descend through fully expanded nodes by the UCT score
        while not node.untried and node.children:
            node = max(node.children, key=lambda ch: uct(ch, node.visits, c))
        # 2. Expansion: add one untried move as a new child
        if node.untried:
            i = node.untried.pop(rng.randrange(len(node.untried)))
            board = play(node.board, i, node.player)
            child = Node(board, other(node.player), node, i)
            node.children.append(child)
            node = child
        # 3. Simulation: play random moves to the end of the game
        b, p = node.board, node.player
        while not winner(b) and "." in b:
            b, p = play(b, rng.choice(legal(b)), p), other(p)
        w = winner(b)
        # 4. Backpropagation: update every node on the path back to the root
        while node:
            node.visits += 1
            mover = other(node.player)
            node.score += 1.0 if w == mover else 0.5 if w is None else 0.0
            node = node.parent
    return max(root.children, key=lambda ch: ch.visits).move   # the most visited move

def match(x_player, o_player, games, seed=0):
    rng, tally = random.Random(seed), {"X": 0, "O": 0, None: 0}
    for _ in range(games):
        b, p = "." * 9, "X"
        while not winner(b) and "." in b:
            move = (x_player if p == "X" else o_player)(b, p, rng)
            b, p = play(b, move, p), other(p)
        tally[winner(b)] += 1
    return {"X wins": tally["X"], "O wins": tally["O"], "draws": tally[None]}

randomly = lambda b, p, rng: rng.choice(legal(b))
perfectly = lambda b, p, rng: perfect_move(b, p)
for n in [20, 200, 1000]:
    player = lambda b, p, rng, n=n: mcts(b, p, n, rng=rng)
    print(f"perfect X vs MCTS({n:4}) as O:", match(perfectly, player, 20))
player = lambda b, p, rng: mcts(b, p, 200, rng=rng)
print("MCTS( 200) as X vs random O: ", match(player, randomly, 20))

## Knowledge and logic

Forward chaining fires every rule whose premises are all known facts, adds the conclusions, and repeats until nothing new appears. Watch what happens in the third case, which no rule anticipated.

In [ ]:
RULES = [
    ({"no_data", "battery_low"}, "power_fault"),
    ({"no_data", "battery_ok"}, "comms_fault"),
    ({"comms_fault", "modem_light_off"}, "replace_modem"),
    ({"power_fault", "panel_dirty"}, "clean_panel"),
    ({"power_fault"}, "send_spare_battery"),
]

def forward_chain(facts, rules):
    facts = set(facts)
    changed = True
    while changed:                 # fire rules until nothing new appears
        changed = False
        for premises, conclusion in rules:
            if premises <= facts and conclusion not in facts:
                facts.add(conclusion)
                changed = True
                print("  ", " and ".join(sorted(premises)), "=>", conclusion)
    return facts

cases = [{"no_data", "battery_ok", "modem_light_off"},
         {"no_data", "battery_low", "panel_dirty"},
         {"no_data", "modem_light_flashing"}]
for facts in cases:
    print(sorted(facts))
    derived = forward_chain(facts, RULES) - facts
    print("   conclusions:", sorted(derived) or "none")

## Reasoning under uncertainty

Bayes' rule (lesson 02) updates a belief with evidence. Bayesian networks make it practical for many variables.

### Bayesian networks

Each variable depends only on its parents, so the joint distribution is a product of small tables: $P(x_1, \ldots, x_n) = \prod_i P(x_i \mid \text{parents}(x_i))$. Inference by enumeration adds up the joint probability of every world consistent with the evidence. Watch the fault probability when the neighbouring station's status is added: this is **explaining away**.

In [ ]:
from itertools import product

# node: (parents, P(node is True | each combination of parent values))
NET = {
    "storm":     ((), {(): 0.10}),
    "fault":     ((), {(): 0.02}),
    "outage":    (("storm", "fault"), {(True, True): 0.95, (True, False): 0.30,
                                       (False, True): 0.90, (False, False): 0.01}),
    "neighbour": (("storm",), {(True,): 0.70, (False,): 0.05}),
}

def joint(world):
    """P(world) as the product of each node's probability given its parents."""
    p = 1.0
    for var, (parents, cpt) in NET.items():
        p_true = cpt[tuple(world[q] for q in parents)]
        p *= p_true if world[var] else 1 - p_true
    return p

def query(var, evidence):
    """P(var is True | evidence), by summing the joint over every possible world."""
    totals = {True: 0.0, False: 0.0}
    for values in product([True, False], repeat=len(NET)):
        world = dict(zip(NET, values))
        if all(world[k] == v for k, v in evidence.items()):
            totals[world[var]] += joint(world)
    return totals[True] / (totals[True] + totals[False])

outage = {"outage": True}
cases = {"P(fault)": {},
         "P(fault | outage)": outage,
         "P(fault | outage, neighbour offline)": {**outage, "neighbour": True},
         "P(fault | outage, neighbour online)": {**outage, "neighbour": False}}
for label, evidence in cases.items():
    print(f"{label:36} = {query('fault', evidence):.3f}")

### Naive Bayes

Score each class with $\log P(c) + \sum_i \log P(w_i \mid c)$, where Laplace smoothing gives $P(w \mid c) = \dfrac{\text{count}(w, c) + \alpha}{\sum_{w'} \text{count}(w', c) + \alpha |V|}$. The data is 24 hand-written maintenance tickets.

In [ ]:
import re
from collections import Counter
import numpy as np

TRAIN = [
    ("battery voltage dropped overnight and the unit shut down", "power"),
    ("solar panel covered in dust, battery not charging", "power"),
    ("power supply fuse blown after the storm", "power"),
    ("unit reboots when the heater switches on, voltage sags", "power"),
    ("battery swollen, replaced and charging again", "power"),
    ("mains adapter failed, running on battery only", "power"),
    ("no charge current from the solar regulator", "power"),
    ("low voltage alarm every night since the cold spell", "power"),
    ("modem not connecting to the network since Tuesday", "comms"),
    ("no data uploaded, cellular signal very weak at the site", "comms"),
    ("SIM card expired so the station stopped reporting", "comms"),
    ("data arrives hours late, upload retries keep failing", "comms"),
    ("router firmware update broke the network connection", "comms"),
    ("antenna cable damaged, signal lost intermittently", "comms"),
    ("server rejects uploads with an authentication error", "comms"),
    ("station offline on the dashboard but power is fine", "comms"),
    ("temperature readings stuck at the same value", "sensor"),
    ("humidity sensor reads above 100 percent", "sensor"),
    ("ozone readings drift upward until recalibration", "sensor"),
    ("filter wheel position error during the morning scan", "sensor"),
    ("pressure sensor noisy, spikes in the readings", "sensor"),
    ("calibration check failed, readings off by ten percent", "sensor"),
    ("spectrometer dark counts much higher than last month", "sensor"),
    ("wind speed sensor bearing worn, readings too low", "sensor"),
]
TEST = [
    ("battery flat again after two cloudy days", "power"),
    ("voltage sags whenever the pump starts", "power"),
    ("fuse replaced but the unit still has no power", "power"),
    ("readings missing whenever the voltage drops", "power"),
    ("no network connection after the modem reboot", "comms"),
    ("uploads failing, signal strength poor", "comms"),
    ("station stopped reporting data to the server", "comms"),
    ("no data since the storm although the battery is fine", "comms"),
    ("readings drift after the filter change", "sensor"),
    ("humidity readings stuck, sensor needs calibration", "sensor"),
    ("temperature sensor spikes every hour", "sensor"),
]

def tokens(text):
    return re.findall(r"\b\w\w+\b", text.lower())   # the same rule scikit-learn uses

class NaiveBayes:
    def fit(self, texts, labels, alpha=1.0):
        self.classes = sorted(set(labels))
        self.vocab = {w for t in texts for w in tokens(t)}
        self.log_prior, self.log_lik = {}, {}
        for c in self.classes:
            docs = [t for t, y in zip(texts, labels) if y == c]
            counts = Counter(w for t in docs for w in tokens(t))
            self.log_prior[c] = np.log(len(docs) / len(texts))
            # Laplace smoothing: pretend every word was seen alpha extra times
            denom = sum(counts.values()) + alpha * len(self.vocab)
            self.log_lik[c] = {w: np.log((counts[w] + alpha) / denom)
                               for w in self.vocab}
        return self

    def scores(self, text):
        """log P(class) + sum of log P(word | class), skipping words never seen."""
        words = [w for w in tokens(text) if w in self.vocab]
        return {c: self.log_prior[c] + sum(self.log_lik[c][w] for w in words)
                for c in self.classes}

    def predict(self, text):
        s = self.scores(text)
        return max(s, key=s.get)

texts, labels = zip(*TRAIN)
nb = NaiveBayes().fit(texts, labels)
preds = [nb.predict(t) for t, _ in TEST]
accuracy = np.mean([p == y for p, (_, y) in zip(preds, TEST)])
print("from scratch accuracy:", round(accuracy, 2))
for (text, y), p in zip(TEST, preds):
    if p != y:
        print(f"  wrong: {text!r} -> {p} (should be {y})")
for c in nb.classes:          # words much more likely in this class than in any other
    lift = {w: nb.log_lik[c][w] - max(nb.log_lik[o][w] for o in nb.classes if o != c)
            for w in sorted(nb.vocab)}
    print(f"  {c:7}", sorted(lift, key=lift.get, reverse=True)[:4])

Scikit-learn's `MultinomialNB` with the same smoothing should agree exactly, down to the scores.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer()
X_train = vec.fit_transform(texts)
X_test = vec.transform([t for t, _ in TEST])
clf = MultinomialNB(alpha=1.0).fit(X_train, labels)
sk_preds = clf.predict(X_test)
print("scikit-learn accuracy:", round(np.mean(sk_preds == [y for _, y in TEST]), 2))
print("same predictions as ours:", list(sk_preds) == preds)

# The scores match too: log P(class) + sum of log P(word | class) for each test ticket
ours = np.array([[nb.scores(t)[c] for c in nb.classes] for t, _ in TEST])
print("same scores as ours:", np.allclose(ours, clf.predict_joint_log_proba(X_test)))

## Planning

In STRIPS, a state is the set of facts that are true, and each action has preconditions, an add list and a delete list. The planner is the breadth-first search from session 1, unchanged.

In [ ]:
from typing import NamedTuple

class Action(NamedTuple):
    name: str
    pre: frozenset                        # facts that must be true beforehand
    add: frozenset                        # facts the action makes true
    delete: frozenset                     # facts the action makes false

def act(name, pre, add, delete=()):
    return Action(name, frozenset(pre), frozenset(add), frozenset(delete))

ACTIONS = [
    act("load spare part", {"at_depot"}, {"have_part"}),
    act("drive to site", {"at_depot"}, {"at_site"}, {"at_depot"}),
    act("drive to depot", {"at_site"}, {"at_depot"}, {"at_site"}),
    act("power down", {"at_site", "powered"}, {"unpowered"}, {"powered"}),
    act("power up", {"at_site", "unpowered"}, {"powered"}, {"unpowered"}),
    act("fit part", {"at_site", "have_part", "unpowered"}, {"part_fitted"},
        {"have_part"}),
    act("calibrate", {"at_site", "powered", "part_fitted"}, {"calibrated"}),
]

def applicable(state):
    for a in ACTIONS:
        if a.pre <= state:
            yield (state - a.delete) | a.add, a.name

def plan(initial, goal):
    """Breadth-first search over sets of facts, with session 1's search code."""
    succ = lambda s: [(s2, 1) for s2, _ in applicable(s)]
    path, expanded = uninformed_search(frozenset(initial), lambda s: goal <= s, succ)
    # Recover the action that links each pair of consecutive states
    names = [next(n for s2, n in applicable(s) if s2 == t)
             for s, t in zip(path, path[1:])]
    return names, expanded

steps, n = plan({"at_depot", "powered"}, {"part_fitted", "calibrated", "at_depot"})
for i, step in enumerate(steps, 1):
    print(i, step)
print(n, "states expanded")

## Classical AI inside LLM agents

Four patterns recur: the LLM models and a solver solves; search over the model's outputs (Tree of Thoughts, best-of-$n$); generate, then verify; and plan, act, check, replan. The exercise builds the verifier that the last two need.

### Hands-on: check a plan before you run it

1. Write `validate(initial, goal, steps)`. It applies the named actions one at a time and returns `(True, "goal reached")`, or `False` with the first step whose preconditions fail and the missing facts.
2. Validate the planner's plan, then this one, as if an LLM had proposed it: `["load spare part", "drive to site", "fit part", "calibrate", "drive to depot"]`.
3. Add an action `log visit` that needs `at_depot` and `calibrated` and adds `logged`. Add `logged` to the goal, replan and validate.

You are done when the proposed plan fails at step 3 because `unpowered` is missing, and the new plan has eight steps.

#### Solution

Simulate the plan against the action definitions, stopping at the first failed precondition.

In [ ]:
def validate(initial, goal, steps):
    """Simulate a plan one step at a time. Return (ok, message)."""
    by_name = {a.name: a for a in ACTIONS}
    state = frozenset(initial)
    for i, name in enumerate(steps, 1):
        if name not in by_name:
            return False, f"step {i}: unknown action {name!r}"
        a = by_name[name]
        missing = a.pre - state
        if missing:
            return False, f"step {i} ({name}): missing {sorted(missing)}"
        state = (state - a.delete) | a.add
    if goal - state:
        return False, f"goal not met: {sorted(goal - state)}"
    return True, "goal reached"

init, goal = {"at_depot", "powered"}, {"part_fitted", "calibrated", "at_depot"}
print("planner's plan:", validate(init, goal, steps))
proposed = ["load spare part", "drive to site", "fit part", "calibrate",
            "drive to depot"]
print("proposed plan: ", validate(init, goal, proposed))

if "log visit" not in [a.name for a in ACTIONS]:       # safe to re-run this cell
    ACTIONS.append(act("log visit", {"at_depot", "calibrated"}, {"logged"}))
new_steps, _ = plan(init, goal | {"logged"})
print("new plan:")
for i, step in enumerate(new_steps, 1):
    print(" ", i, step)
print("new plan check:", validate(init, goal | {"logged"}, new_steps))

**What to observe.** The model can propose a plan in any wording, but before anything touches real equipment a dozen lines of deterministic code check it against a model of the world. The error message is exactly what you would send back to the model so it can repair its plan: a guess turned into a search with a verifier.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/24-classical-ai.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.